# Flow Matching: Space Mapping (CSTR Design)

Map desired concentrations $(C_a, C_b)$ to design parameters $(R, RT)$ for a CSTR.

**Authors:** Victor Alves and John R. Kitchin

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import warnings
import os

# Force CPU for JAX (must be set before importing JAX)
os.environ['JAX_PLATFORMS'] = 'cpu'

import torch

# Import reusable utilities from local module
from generative_optimization import (
    generate_samples,
    cluster_stats,
    ConditionalFlowMatching
)

# Force CPU for PyTorch
device = torch.device('cpu')
print(f"Using device: {device}")

# Figure settings
mpl.rcParams['figure.facecolor'] = 'white'
mpl.rcParams['axes.facecolor'] = 'white'
mpl.rcParams['figure.dpi'] = 150

warnings.filterwarnings('ignore')

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

## Problem Setup

CSTR with reactions:
$$2A \xrightarrow{k_1} B$$
$$B \xrightarrow{k_2} C$$

Design parameters:
- $R$: reactor radius
- $RT$: dimensionless temperature parameter

Outputs:
- $C_a$: exit concentration of A
- $C_b$: exit concentration of B

**Inverse problem**: Given desired $(C_a, C_b)$, find $(R, RT)$.

In [ ]:
# CSTR model
H = 1
vo = 1
Cao = 1

bounds = [[0.1, 3], [0.1, 15]]
X = generate_samples(bounds, n_samples=1024, seed=42)

R, RT = X.T
V = np.pi * R**2 * H
k1 = np.exp(-3 / RT)
k2 = np.exp(-10 / RT)

a = V * k1
b = vo + V * k2
c = -vo * Cao
Ca = (-b + np.sqrt(b**2 - 4*a*c)) / (2*a)
Cb = k1 * Ca**2 * V / vo

print(f"Ca range: [{Ca.min():.3f}, {Ca.max():.3f}]")
print(f"Cb range: [{Cb.min():.3f}, {Cb.max():.3f}]")

In [ ]:
# Train: generate [R, RT] conditioned on [Ca, Cb]
x_data = np.column_stack([R, RT])
c_data = np.column_stack([Ca, Cb])

fm_map = ConditionalFlowMatching(x_dim=2, c_dim=2, hidden_dim=128, n_layers=4)
losses = fm_map.fit(x_data, c_data, epochs=1000, batch_size=64)

In [ ]:
# Map desired output region to input space
center = (0.58, 0.3)
radius = 0.08

# Generate points in desired circle
theta = np.random.uniform(0, 2*np.pi, 500)
r = np.sqrt(np.random.uniform(0, 1, 500)) * radius
Ca_desired = center[0] + r * np.cos(theta)
Cb_desired = center[1] + r * np.sin(theta)

# Map to input space
c_values = np.column_stack([Ca_desired, Cb_desired])
mapped = []
for c in c_values:
    s = fm_map.sample(c_values=[c], n_samples=1)
    mapped.append(s[0])
mapped = np.array(mapped)

print(f"Mapped {len(mapped)} points")

In [ ]:
# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = np.sqrt(R**2 + RT**2)

# Input space
axes[0].scatter(R, RT, s=4, c=colors, cmap='viridis', alpha=0.5)
axes[0].scatter(mapped[:, 0], mapped[:, 1], c='red', s=10, alpha=0.3, 
                label='Mapped region')
axes[0].set_xlabel('R')
axes[0].set_ylabel('RT')
axes[0].set_title('Input space')
axes[0].legend()

# Output space
axes[1].scatter(Ca, Cb, s=4, c=colors, cmap='viridis', alpha=0.5)
axes[1].scatter(Ca_desired, Cb_desired, c='red', s=10, alpha=0.3,
                label='Desired region')
circle = plt.Circle(center, radius, fill=False, color='red', ls='--')
axes[1].add_patch(circle)
axes[1].set_xlabel('$C_a$')
axes[1].set_ylabel('$C_b$')
axes[1].set_title('Output space')
axes[1].axis('equal')
axes[1].legend()

plt.tight_layout()
plt.show()